# PLAsTiCC Data Glance

This notebook gives a quick overview of data size and structure for files in the `data/` folder.

In [3]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

In [4]:
DATA_DIR = Path('data')
csv_files = sorted(DATA_DIR.glob('*.csv'))
csv_files

[PosixPath('data/plasticc_test_metadata.csv'),
 PosixPath('data/plasticc_test_set_batch1.csv'),
 PosixPath('data/plasticc_test_set_batch10.csv'),
 PosixPath('data/plasticc_test_set_batch11.csv'),
 PosixPath('data/plasticc_test_set_batch2.csv'),
 PosixPath('data/plasticc_test_set_batch3.csv'),
 PosixPath('data/plasticc_test_set_batch4.csv'),
 PosixPath('data/plasticc_test_set_batch5.csv'),
 PosixPath('data/plasticc_test_set_batch6.csv'),
 PosixPath('data/plasticc_test_set_batch7.csv'),
 PosixPath('data/plasticc_test_set_batch8.csv'),
 PosixPath('data/plasticc_test_set_batch9.csv'),
 PosixPath('data/plasticc_train_lightcurves.csv'),
 PosixPath('data/plasticc_train_metadata.csv')]

In [5]:
def human_mb(num_bytes: int) -> float:
    return num_bytes / (1024 ** 2)

def csv_shape_fast(path: Path, chunksize: int = 200_000):
    # Compute rows/cols with chunked read to avoid huge memory use.
    rows = 0
    cols = None
    for i, chunk in enumerate(pd.read_csv(path, chunksize=chunksize, low_memory=False)):
        rows += len(chunk)
        if i == 0:
            cols = len(chunk.columns)
    return rows, (cols if cols is not None else 0)

summary_rows = []
for f in csv_files:
    n_rows, n_cols = csv_shape_fast(f)
    summary_rows.append({
        'file': f.name,
        'size_mb': round(human_mb(f.stat().st_size), 2),
        'rows': n_rows,
        'cols': n_cols,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('size_mb', ascending=False).reset_index(drop=True)
summary_df

,file,size_mb,rows,cols
0,plasticc_test_set_batch10.csv,1881.26,44282019,6
1,plasticc_test_set_batch11.csv,1880.74,44268608,6
2,plasticc_test_set_batch9.csv,1854.78,44282427,6
3,plasticc_test_set_batch4.csv,1840.08,44304969,6
4,plasticc_test_set_batch8.csv,1839.23,44285107,6
5,plasticc_test_set_batch5.csv,1839.00,44278664,6
6,plasticc_test_set_batch6.csv,1838.65,44273088,6
7,plasticc_test_set_batch3.csv,1838.58,44271312,6
8,plasticc_test_set_batch7.csv,1838.57,44269257,6
9,plasticc_test_set_batch2.csv,1809.90,44281695,6


In [6]:
# Structure view: columns and inferred dtypes (sample-based)
structure_rows = []
for f in csv_files:
    sample = pd.read_csv(f, nrows=2000, low_memory=False)
    structure_rows.append({
        'file': f.name,
        'n_columns': len(sample.columns),
        'columns': list(sample.columns),
        'dtypes_sample': {c: str(sample[c].dtype) for c in sample.columns}
    })

structure_df = pd.DataFrame(structure_rows)
structure_df[['file', 'n_columns']]

,file,n_columns
0,plasticc_test_metadata.csv,26
1,plasticc_test_set_batch1.csv,6
2,plasticc_test_set_batch10.csv,6
3,plasticc_test_set_batch11.csv,6
4,plasticc_test_set_batch2.csv,6
5,plasticc_test_set_batch3.csv,6
6,plasticc_test_set_batch4.csv,6
7,plasticc_test_set_batch5.csv,6
8,plasticc_test_set_batch6.csv,6
9,plasticc_test_set_batch7.csv,6


In [9]:
# Detailed per-file structure
for _, row in structure_df.iterrows():
    print('=' * 120)
    print(f"FILE: {row['file']}")
    print(f"N_COLUMNS: {row['n_columns']}")
    print('COLUMNS:')
    print(row['columns'])
    print('DTYPES (sample-based):')
    print(row['dtypes_sample'])
    print()

FILE: plasticc_test_metadata.csv
N_COLUMNS: 26
COLUMNS:
['object_id', 'ra', 'decl', 'ddf_bool', 'hostgal_specz', 'hostgal_photoz', 'hostgal_photoz_err', 'distmod', 'mwebv', 'target', 'true_target', 'true_submodel', 'true_z', 'true_distmod', 'true_lensdmu', 'true_vpec', 'true_rv', 'true_av', 'true_peakmjd', 'libid_cadence', 'tflux_u', 'tflux_g', 'tflux_r', 'tflux_i', 'tflux_z', 'tflux_y']
DTYPES (sample-based):
{'object_id': 'int64', 'ra': 'float64', 'decl': 'float64', 'ddf_bool': 'int64', 'hostgal_specz': 'float64', 'hostgal_photoz': 'float64', 'hostgal_photoz_err': 'float64', 'distmod': 'float64', 'mwebv': 'float64', 'target': 'int64', 'true_target': 'int64', 'true_submodel': 'int64', 'true_z': 'float64', 'true_distmod': 'float64', 'true_lensdmu': 'float64', 'true_vpec': 'float64', 'true_rv': 'float64', 'true_av': 'float64', 'true_peakmjd': 'float64', 'libid_cadence': 'int64', 'tflux_u': 'float64', 'tflux_g': 'float64', 'tflux_r': 'float64', 'tflux_i': 'float64', 'tflux_z': 'float64',

In [11]:
# Preview key files
key_files = [
    DATA_DIR / 'plasticc_train_metadata.csv',
    DATA_DIR / 'plasticc_train_lightcurves.csv',
    DATA_DIR / 'plasticc_test_metadata.csv',
]

for f in key_files:
    if f.exists():
        print('=' * 120)
        print(f"Preview: {f.name}")
        display(pd.read_csv(f, nrows=5, low_memory=False))

Preview: plasticc_train_metadata.csv


,object_id,ra,decl,ddf_bool,hostgal_specz,hostgal_photoz,hostgal_photoz_err,distmod,mwebv,target,true_target,true_submodel,true_z,true_distmod,true_lensdmu,true_vpec,true_rv,true_av,true_peakmjd,libid_cadence,tflux_u,tflux_g,tflux_r,tflux_i,tflux_z,tflux_y
0,615,349.0461,-61.9438,1,0.000,0.000,0.000,-9.000,0.017,92,92,1,0.000,0.000,0.000,0.0,0.0,0.0,59570.000,69,484.7,3286.7,3214.1,3039.7,2854.5,2837.0
1,713,53.0859,-27.7844,1,1.818,1.627,0.255,45.406,0.007,88,88,1,1.817,45.703,0.000,0.0,0.0,0.0,59570.000,34,108.7,117.7,119.9,149.6,147.9,150.5
2,730,33.5742,-6.5796,1,0.232,0.226,0.016,40.256,0.021,42,42,2,0.233,40.328,0.004,4.5,0.0,0.0,60444.379,9,0.0,0.0,0.0,0.0,0.0,0.0
3,745,0.1899,-45.5867,1,0.304,0.281,1.152,40.795,0.007,90,90,1,0.301,40.969,-0.004,257.7,0.0,0.0,60130.453,38,0.0,0.0,0.0,0.0,0.0,0.0
4,1124,352.7113,-63.8237,1,0.193,0.241,0.018,40.417,0.024,90,90,1,0.193,39.866,-0.002,-368.8,0.0,0.0,60452.641,1,0.0,0.0,0.0,0.0,0.0,0.0


Preview: plasticc_train_lightcurves.csv


,object_id,mjd,passband,flux,flux_err,detected_bool
0,615,59750.4229,2,-544.810303,3.622952,1
1,615,59750.4306,1,-816.434326,5.553370,1
2,615,59750.4383,3,-471.385529,3.801213,1
3,615,59750.4450,4,-388.984985,11.395031,1
4,615,59752.4070,2,-681.858887,4.041204,1


Preview: plasticc_test_metadata.csv


,object_id,ra,decl,ddf_bool,hostgal_specz,hostgal_photoz,hostgal_photoz_err,distmod,mwebv,target,true_target,true_submodel,true_z,true_distmod,true_lensdmu,true_vpec,true_rv,true_av,true_peakmjd,libid_cadence,tflux_u,tflux_g,tflux_r,tflux_i,tflux_z,tflux_y
0,13,34.4531,-5.2295,1,0.305,0.319,0.054,41.112,0.019,0,42,1,0.302,40.977,0.000,120.2,0.0,0.0,60499.461,124,0.0,0.0,0.0,0.0,0.0,0.0
1,14,33.3984,-4.3311,1,-9.000,0.632,0.018,42.877,0.018,0,42,2,0.610,42.785,-0.027,584.6,0.0,0.0,59792.121,120,0.0,0.0,0.0,0.0,0.0,0.0
2,17,348.5294,-61.7554,1,-9.000,0.830,0.060,43.600,0.016,0,42,2,0.826,43.587,0.001,-148.0,0.0,0.0,60543.566,85,0.0,0.0,0.0,0.0,0.0,0.0
3,23,34.8047,-5.8292,1,-9.000,0.653,0.148,42.964,0.023,0,90,1,0.624,42.842,0.010,-3.5,0.0,0.0,60137.480,97,0.0,0.0,0.0,0.0,0.0,0.0
4,34,351.3214,-64.1987,1,0.456,0.462,0.012,42.054,0.023,0,90,1,0.454,42.015,-0.047,261.6,0.0,0.0,60245.078,68,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Optional: check object_id coverage (train metadata vs train lightcurves)
train_meta_path = DATA_DIR / 'plasticc_train_metadata.csv'
train_lc_path = DATA_DIR / 'plasticc_train_lightcurves.csv'

if train_meta_path.exists() and train_lc_path.exists():
    train_meta_ids = pd.read_csv(train_meta_path, usecols=['object_id'])['object_id'].nunique()
    train_lc_ids = pd.read_csv(train_lc_path, usecols=['object_id'])['object_id'].nunique()
    print(f\"Unique object_id in train metadata:    {train_meta_ids:,}\")
    print(f\"Unique object_id in train lightcurve:   {train_lc_ids:,}\")